# Klasifikasi Risiko Pengunduran Diri Karyawan Berdasarkan Data HR Menggunakan Random Forest
**Dataset:** IBM HR Analytics Employee Attrition & Performance (`WA_Fn-UseC_-HR-Employee-Attrition.csv`)  
**Algoritma:** Random Forest Classifier (Ensemble Learning)

---

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    precision_recall_curve, roc_curve
)

sns.set_style('whitegrid')

## 1. Load Dataset & Exploratory Data Analysis (EDA)

In [ ]:
# Menentukan path dataset secara dinamis agar bisa berjalan baik di Google Colab, VS Code, maupun Jupyter Notebook
candidate_paths = [
    '../data/WA_Fn-UseC_-HR-Employee-Attrition.csv',
    'data/WA_Fn-UseC_-HR-Employee-Attrition.csv',
    'WA_Fn-UseC_-HR-Employee-Attrition.csv'
]

dataset_path = None
for path in candidate_paths:
    if os.path.exists(path):
        dataset_path = path
        break

if dataset_path is None:
    raise FileNotFoundError("File WA_Fn-UseC_-HR-Employee-Attrition.csv tidak ditemukan di folder '../data/', 'data/', atau folder saat ini.")

print(f"Memuat dataset dari: {dataset_path}")
df = pd.read_csv(dataset_path)
print('Ukuran Dataset:', df.shape)
df.head()

In [ ]:
print('Jumlah Missing Value:', df.isnull().sum().sum())
print('\nDistribusi Target Attrition:')
print(df['Attrition'].value_counts())
print('\nPersentase Target Attrition (%):')
print((df['Attrition'].value_counts(normalize=True) * 100).round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.countplot(data=df, x='Attrition', palette=['#1e293b', '#0f766e'], ax=axes[0])
axes[0].set_title('Distribusi Kelas Attrition')

sns.countplot(data=df, x='OverTime', hue='Attrition', palette=['#1e293b', '#0f766e'], ax=axes[1])
axes[1].set_title('Attrition Berdasarkan Lembur (OverTime)')

sns.boxplot(data=df, x='Attrition', y='MonthlyIncome', palette=['#1e293b', '#0f766e'], ax=axes[2])
axes[2].set_title('Distribusi Pendapatan Bulanan vs Attrition')

plt.tight_layout()
plt.show()

## 2. Data Preprocessing & Feature Engineering

In [ ]:
# Menghapus kolom ID dan kolom bernilai konstan
drop_cols = ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
df_clean = df.drop(columns=drop_cols).copy()
df_clean['Attrition_Binary'] = df_clean['Attrition'].map({'Yes': 1, 'No': 0})

X = df_clean.drop(columns=['Attrition', 'Attrition_Binary'])
y = df_clean['Attrition_Binary']

categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(exclude=['object']).columns.tolist()

print(f'Jumlah Fitur Numerik: {len(numerical_cols)}, Fitur Kategorikal: {len(categorical_cols)}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

## 3. Pemodelan Random Forest Classifier & Validasi Silang

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ]
)

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_split=8,
        min_samples_leaf=4,
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(rf_pipeline, X_train, y_train, cv=cv, scoring='roc_auc')
print(f'5-Fold Cross Validation ROC-AUC: {cv_auc.mean():.4f} (+/- {cv_auc.std():.4f})')

## 4. Evaluasi Model, Kurva ROC, dan Feature Importance

In [ ]:
y_proba = rf_pipeline.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_thresh = float(thresholds[np.argmax(f1_scores)])
y_pred = (y_proba >= best_thresh).astype(int)

print(f'Threshold Optimal : {best_thresh:.4f}')
print(f'Accuracy          : {accuracy_score(y_test, y_pred)*100:.2f}%')
print(f'Precision         : {precision_score(y_test, y_pred)*100:.2f}%')
print(f'Recall            : {recall_score(y_test, y_pred)*100:.2f}%')
print(f'F1-Score          : {f1_score(y_test, y_pred)*100:.2f}%')
print(f'ROC-AUC           : {roc_auc_score(y_test, y_proba):.4f}')
print('\nClassification Report:\n', classification_report(y_test, y_pred, target_names=['Bertahan (No)', 'Resign (Yes)']))

In [ ]:
# Visualisasi Confusion Matrix & ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='GnBu', cbar=False,
            xticklabels=['Bertahan', 'Resign'], yticklabels=['Bertahan', 'Resign'], ax=axes[0])
axes[0].set_title(f'Confusion Matrix (Threshold = {best_thresh:.2f})')
axes[0].set_xlabel('Prediksi')
axes[0].set_ylabel('Aktual')

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='#0f766e', lw=2.5, label=f'ROC AUC = {roc_auc_score(y_test, y_proba):.3f}')
axes[1].plot([0, 1], [0, 1], '--', color='gray')
axes[1].set_title('Kurva ROC Random Forest')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Ekspor model terlatih untuk deployment ke aplikasi Flask
model_bundle = {
    'pipeline': rf_pipeline,
    'threshold': best_thresh,
    'feature_columns': list(X.columns)
}

save_paths = ['rf_attrition_model.joblib']
if os.path.exists('../models'):
    save_paths.append('../models/rf_attrition_model.joblib')
elif os.path.exists('models'):
    save_paths.append('models/rf_attrition_model.joblib')

for p in save_paths:
    joblib.dump(model_bundle, p)
print(f"File model berhasil disimpan ke: {', '.join(save_paths)}")